In [1]:
import sys

assert sys.version_info>= (3,10)

In [2]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

This chapter can be very slow without a hardware accelerator, so if we can find one, let's use it:

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [4]:
import matplotlib.pyplot as plt

plt.rc("font", size=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("legend", fontsize = 14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

Let's use the same evaluate_tm() and train() functions as in the previous chapters:


In [5]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader,
          valid_loader, n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [],
               "train_metrics": [],
               "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred , y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/ (index + 1)}:.4f", end="")
            print(f" {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch: {epoch + 1}/{n_epochs},"
              f"train loss: {history["train_losses"][-1]:.4f},"
              f"train metric: {history["train_metrics"][-1]:.2%},"
              f"valid metric: {history["valid_metrics"][-1]:.2%}")
    return history
        

/home/ziaur/ziazh/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


As we will build and download pretty big models, we will need to free the GPU RAM regularly to avoid running ouut of space. For this, we will delete the models and tensors as we go, using the del_vars() function below. it deletes the given variables(if they exist) then calls Python's garbage collector, and it also calls torch.cuda.empty_cache() if we're usinga CUDA GPU:

In [6]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

Generatin Shakespearean Text Using a Character RNN

Creating the Training Dataset

Let's download the Shakespeare data from Andrej Karpathy's

In [7]:
from pathlib import Path
import urllib.request

def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

shakespeare_text = download_shakespeare_text()

In [8]:
print(shakespeare_text[:80])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.


In [9]:
vocab = sorted(set(shakespeare_text.lower()))
"".join(vocab)

"\n !$&',-.3:;?abcdefghijklmnopqrstuvwxyz"

In [10]:
char_to_id = {char: index for index, char in enumerate(vocab)}
id_to_char = {index: char for index, char in enumerate(vocab)}


In [11]:
char_to_id["a"]

13

In [12]:
id_to_char[13]

'a'

In [13]:
def encode_text(text):
    return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(char_ids):
    return "".join([id_to_char[char_id.item()]for char_id in char_ids])

In [14]:
encoded = encode_text("hello, world!")
encoded

tensor([20, 17, 24, 24, 27,  6,  1, 35, 27, 30, 24, 16,  2])

In [15]:
decode_text(encoded)

'hello, world!'

In [16]:
from torch.utils.data import Dataset, DataLoader

class CharDataset(Dataset):
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) -  self.window_length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        end = idx + self.window_length
        window = self.encoded_text[idx : end]
        target = self.encoded_text[idx + 1: end + 1]
        return window, target

In [17]:
to_be_dataset  = CharDataset("To be of not to be", window_length=10)
for x, y in to_be_dataset:
    print(f"x={x}, y={y}")
    print(f"    decoded: x={decode_text(x)!r}, y={decode_text(y)!r}")

x=tensor([32, 27,  1, 14, 17,  1, 27, 18,  1, 26]), y=tensor([27,  1, 14, 17,  1, 27, 18,  1, 26, 27])
    decoded: x='to be of n', y='o be of no'
x=tensor([27,  1, 14, 17,  1, 27, 18,  1, 26, 27]), y=tensor([ 1, 14, 17,  1, 27, 18,  1, 26, 27, 32])
    decoded: x='o be of no', y=' be of not'
x=tensor([ 1, 14, 17,  1, 27, 18,  1, 26, 27, 32]), y=tensor([14, 17,  1, 27, 18,  1, 26, 27, 32,  1])
    decoded: x=' be of not', y='be of not '
x=tensor([14, 17,  1, 27, 18,  1, 26, 27, 32,  1]), y=tensor([17,  1, 27, 18,  1, 26, 27, 32,  1, 32])
    decoded: x='be of not ', y='e of not t'
x=tensor([17,  1, 27, 18,  1, 26, 27, 32,  1, 32]), y=tensor([ 1, 27, 18,  1, 26, 27, 32,  1, 32, 27])
    decoded: x='e of not t', y=' of not to'
x=tensor([ 1, 27, 18,  1, 26, 27, 32,  1, 32, 27]), y=tensor([27, 18,  1, 26, 27, 32,  1, 32, 27,  1])
    decoded: x=' of not to', y='of not to '
x=tensor([27, 18,  1, 26, 27, 32,  1, 32, 27,  1]), y=tensor([18,  1, 26, 27, 32,  1, 32, 27,  1, 14])
    decoded: x=

In [18]:
window_length = 50
batch_size = 512
train_set = CharDataset(shakespeare_text[:1_000_000], window_length)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_set = CharDataset(shakespeare_text[1_000_000:1_060_000], window_length)
valid_loader   = DataLoader(valid_set, batch_size=batch_size)
test_set = CharDataset(shakespeare_text[1_060_000:], window_length)
test_loader = DataLoader(test_set, batch_size=batch_size)


Embeddings

In [19]:
import torch.nn as nn 

torch.manual_seed(42)
embed = nn.Embedding(5, 3)
embed(torch.tensor([[3,2], [0, 2]]))

tensor([[[ 0.2674,  0.5349,  0.8094],
         [ 2.2082, -0.6380,  0.4617]],

        [[ 0.3367,  0.1288,  0.2345],
         [ 2.2082, -0.6380,  0.4617]]], grad_fn=<EmbeddingBackward0>)

Building and Training the Char-RNN Model

Now let's create out Shakespeare model:

In [20]:
class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128,
                 dropout=0.1):
        super().__init__()
        self.embed= nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0, 2, 1)

torch.manual_seed(42)
model = ShakespeareModel(len(vocab)).to(device)

In [21]:
n_epochs = 2
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=len(vocab)).to(device)

# history = train(model, optimizer, xentropy, accuracy, train_loader,
                # valid_loader,n_epochs)

In [22]:
torch.save(model.state_dict(), "my_shakespeare_model.pt")

In [23]:
model.eval()
text = "To be or not to b"
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
with torch.no_grad():
    Y_logits = model(encoded_text)
    predicted_char_id = Y_logits[0, :, -1].argmax().item()
    predicted_char  = id_to_char[predicted_char_id]

In [24]:
predicted_char

'3'

Generating Shakespearean Text

In [25]:
torch.manual_seed(42)
probs = torch.tensor([[0.5, 0.4, 0.1]])
samples = torch.multinomial(probs, replacement=True, num_samples=8)
samples

tensor([[0, 0, 0, 0, 1, 0, 2, 2]])

In [26]:
import torch.nn.functional as F 

def next_char(model, text, temperature=1):
    encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
    with torch.no_grad():
        Y_logits = model(encoded_text)
        Y_probas = F.softmax(Y_logits[0, :, -1] / temperature, dim=-1)
        predicted_char_id = torch.multinomial(Y_probas, num_samples=1).item()

    return id_to_char[predicted_char_id]

In [27]:
def extend_text(model, text, n_chars=80, temperature=1):
    for _ in range(n_chars):
        text += next_char(model, text, temperature)

    return text

In [28]:
print(extend_text(model, text="To be or not to b", temperature=0.01))

To be or not to b3f33?????px? ppbbxx pf$bf333?$3?3?????p?xp ??$pxp bf33?p??p? $333??3??????p???$?


In [29]:
print(extend_text(model, "To be or not to b", temperature=0.4))

To be or not to bp$yacd.r3pgauypj.wgitcea3,qk?tm
jr.$pe&xhk&bnecf;.onif
'&yt.y&3u.hdhfmb,,zmtffe:


In [30]:
print(extend_text(model, "To be or not to b", temperature=100))

To be or not to bmhf:my:rtk;s-h cqvvnfnfsut&-oq'ryoeen?x-hp:d,y&wv f3,dzrdzj-p$lv?xpzc,fborp;'?$u


In [31]:
class StatefullShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128,
                 dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, vocab_size)
        self.hidden_states = None

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, hidden_states = self.gru(embeddings, self.hidden_states)
        self.hidden_states = hidden_states.detach()
        return self.output(outputs).permute(0, 2, 1)
        
        

In [32]:
from torch.utils.data import Dataset, DataLoader

class StatefullCharDataset(Dataset):
    def __init__(self, text, window_length, batch_size):
        self.encoded_text = encode_text(text)
        self.window_length  = window_length
        self.batch_size = batch_size
        n_consecutive_windows = (len(self.encoded_text) - 1) // window_length
        n_windows_per_slot = n_consecutive_windows // batch_size
        self.length = n_windows_per_slot * batch_size
        self.spacing = n_windows_per_slot * window_length

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError("dataset index out of range")
        start = ((idx % self.batch_size) * self.spacing
                 + (idx // self.batch_size) * self.window_length)
        end = start + self.window_length
        window = self.encoded_text[start: end]
        target = self.encoded_text[start + 1: end + 1]
        return window, target

In [33]:
batch_size = 128
stateful_train_set = StatefullCharDataset(shakespeare_text[:1_000_000], window_length=window_length, batch_size=batch_size)
stateful_train_loader = DataLoader(stateful_train_set, batch_size, drop_last=True)
statful_valid_set   = StatefullCharDataset(shakespeare_text[1_000_000:1_060_000], 
                                           window_length, batch_size,)
stateful_valid_loader = DataLoader(statful_valid_set, batch_size, drop_last=True)
stateful_test_set = StatefullCharDataset(shakespeare_text[1_060_000:],
                                         window_length, batch_size)
stateful_test_loader  = DataLoader(stateful_test_set, batch_size, drop_last=True)

In [34]:
torch.manual_seed(42)
stateful_model = StatefullShakespeareModel(len(vocab)).to(device)

n_epochs=1
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=len(vocab)).to(device)
def reset_hidden_states(model, epoch):
    model.hidden_states = None

# history = train(stateful_model, optimizer, xentropy, accuracy, stateful_train_loader,
#                 stateful_valid_loader, n_epochs, epoch_callback=reset_hidden_states)

In [35]:
Out.clear()  # clear Jupyter's `Out` variable which saves all the cell outputs
del_vars(["accuracy", "embed", "encoded", "encoded_text", "optimizer", "probs",
          "samples", "x", "y", "shakespeare_text", "stateful_test_loader",
          "stateful_train_loader", "Y_logits", "stateful_valid_loader",
          "test_loader", "train_loader", "valid_loader", "xentropy"])

Loading the IMDB Dataset

In [36]:
from datasets import load_dataset

imdb_dataset = load_dataset("stanfordnlp/imdb")
split = imdb_dataset["train"].train_test_split(train_size=0.8, seed=42)
imdb_train_set, imdb_valid_set = split["train"], split["test"]
imdb_test_set = imdb_dataset["test"]

In [37]:
imdb_train_set[1]["text"]

"'The Rookie' was a wonderful movie about the second chances life holds for us and also puts an emotional thought over the audience, making them realize that your dreams can come true. If you loved 'Remember the Titans', 'The Rookie' is the movie for you!! It's the feel good movie of the year and it is the perfect movie for all ages. 'The Rookie' hits a major home run!"

Tokenization Using the tokenizers Library

Training a BPE Tokenizer on the IMDB Dataset

In [38]:
import tokenizers

bpe_model = tokenizers.models.BPE(unk_token="<unk>")
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
special_tokens = ["<pad>", "<unk>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000,
                                             special_tokens=special_tokens)
train_reviews = [review["text"].lower() for review in imdb_train_set]
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

In [39]:
tokenizers.pre_tokenizers.Whitespace().pre_tokenize_str("Hello, world!")

[('Hello', (0, 5)), (',', (5, 6)), ('world', (7, 12)), ('!', (12, 13))]

Encoding and Decoding Text

In [40]:
some_review = "what an awesome movie! 😊"
bpe_encoding = bpe_tokenizer.encode(some_review)
bpe_encoding

Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [41]:
bpe_encoding.tokens

['what', 'an', 'aw', 'es', 'ome', 'movie', '!', '<unk>']

In [42]:
bpe_token_ids = bpe_encoding.ids
bpe_token_ids

[303, 139, 373, 149, 240, 211, 4, 1]

In [43]:
bpe_tokenizer.get_vocab()["what"]

303

In [44]:
bpe_tokenizer.id_to_token(305)

'ough'

Handling batches

In [45]:
bpe_tokenizer.encode_batch(train_reviews[:3])

[Encoding(num_tokens=281, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=114, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=285, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

In [46]:
bpe_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=500)

In [47]:
bpe_encodings = bpe_tokenizer.encode_batch(train_reviews[:3])
bpe_batch_ids = torch.tensor([encoding.ids for encoding in bpe_encodings])
bpe_batch_ids

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

In [48]:
attention_mask = torch.tensor([encoding.attention_mask for encoding in bpe_encodings])
attention_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [49]:
lengths = attention_mask.sum(dim=1)
lengths

tensor([281, 114, 285])

BBPE Tokenization

The emoji is represented as 4 bytes when usin the UTF-8 Unicode encoding, so the ByteLevel pre-tokenizer represents it as 4 characters, each representing a byte. Spaces are converted t G

In [50]:
tokenizers.pre_tokenizers.ByteLevel().pre_tokenize_str(some_review)

[('Ġwhat', (0, 4)),
 ('Ġan', (4, 7)),
 ('Ġawesome', (7, 15)),
 ('Ġmovie', (15, 21)),
 ('!', (21, 22)),
 ('ĠðŁĺĬ', (22, 24))]

In [51]:
bbpe_model = tokenizers.models.BPE(unk_token="<unk>")
bbpe_tokenizer = tokenizers.Tokenizer(bbpe_model)
bbpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
bbpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000,special_tokens=special_tokens)
bbpe_tokenizer.train_from_iterator(train_reviews, bbpe_trainer)

In [52]:
bbpe_encoding = bbpe_tokenizer.encode(some_review)
bbpe_tokens = bbpe_encoding.tokens
bbpe_tokens

['Ġwhat',
 'Ġan',
 'Ġaw',
 'es',
 'ome',
 'Ġmovie',
 '!',
 'Ġ',
 '<unk>',
 'Ł',
 'ĺ',
 '<unk>']

In [53]:
bbpe_token_ids = bbpe_encoding.ids
bbpe_token_ids

[354, 216, 561, 148, 244, 232, 2, 107, 1, 125, 119, 1]

In [54]:
bbpe_decoded = bbpe_tokenizer.decode(bbpe_token_ids)
bbpe_decoded

'Ġwhat Ġan Ġaw es ome Ġmovie ! Ġ Ł ĺ'

In [55]:
bbpe_decoded.replace(" ", "").replace("Ġ"," ").strip()

'what an awesome movie! Łĺ'

WorkPiece

In [56]:
wp_model = tokenizers.models.WordPiece(unk_token="<unk>")
wp_tokenizer = tokenizers.Tokenizer(wp_model)
wp_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
wp_trainer  = tokenizers.trainers.WordPieceTrainer(vocab_size=1000,
                                                   special_tokens=special_tokens)
wp_tokenizer.train_from_iterator(train_reviews, wp_trainer)


In [57]:
wp_encoding = wp_tokenizer.encode(some_review)
wp_tokens = wp_encoding.tokens
wp_tokens

['what', 'an', 'aw', '##es', '##ome', 'movie', '!', '<unk>']

In [58]:
wp_token_ids = wp_encoding.ids
wp_token_ids

[443, 312, 635, 257, 354, 331, 4, 1]

In [59]:
wp_decoded = wp_tokenizer.decode(wp_token_ids)
wp_decoded

'what an aw ##es ##ome movie !'

In [60]:
wp_decoded.replace(" ##", "").replace(" !", "!")

'what an awesome movie!'

Unigram LM

In [61]:
unigram_model = tokenizers.models.Unigram()
unigram_tokenizer = tokenizers.Tokenizer(unigram_model)
unigram_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
unigram_trainer = tokenizers.trainers.UnigramTrainer(
    vocab_size=1000, special_tokens=special_tokens, unk_token="<unk>")
unigram_tokenizer.train_from_iterator(train_reviews, unigram_trainer)

In [62]:
unigram_encoding = unigram_tokenizer.encode(some_review)
unigram_tokens = unigram_encoding.tokens
unigram_tokens

['what', 'an', 'a', 'w', 'e', 'some', 'movie', '!', '😊']

In [63]:
unigram_token_ids = unigram_encoding.ids
unigram_token_ids

[79, 37, 4, 40, 6, 70, 46, 74, 1]

In [64]:
unigram_decoded = unigram_tokenizer.decode(unigram_token_ids)
unigram_decoded

'what an a w e some movie !'

Pretrained Tokenizers

BBPE is used by models like GPT-2 and RoBERTa.

In [65]:
import transformers

gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_encoding = gpt2_tokenizer(train_reviews[:3], truncation=True,
                               max_length=500)

In [66]:
gpt2_encoding.keys()

KeysView({'input_ids': [[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690, 1282, 503, 2045, 588, 257, 2646, 4676, 373, 2391, 4624, 319, 262, 3800, 357, 10508, 355, 366, 3847, 2802, 11074, 9785, 1681, 46390, 316, 338, 4571, 7622, 262, 2646, 6776, 11, 543, 318, 2592, 2408, 1201, 262, 4286, 4438, 683, 645, 1103, 4427, 13, 991, 11, 340, 338, 3621, 284, 804, 379, 329, 644, 340, 318, 13, 262, 16585, 1022, 285, 40302, 269, 5718, 290, 33826, 8803, 302, 44655, 318, 2407, 10457, 13, 262, 17262, 286, 511, 2776, 389, 6452, 13, 269, 5718, 318, 9623, 355, 1464, 11, 290, 302, 44655, 3011, 530, 286, 465, 1178, 8395, 284, 1107, 719, 29847, 1671, 1220, 6927, 1671, 11037, 72, 22127, 326, 1312, 1053, 1239, 1775, 4173, 64, 443, 7114, 338, 711, 11, 475, 1312, 3285, 326, 474, 323, 1803, 261, 477, 268, 338, 16711, 318, 17074, 13, 262, 4226, 318, 8131, 47370, 11, 290, 7622, 345, 25260, 13, 366, 22595, 46670, 1, 318, 281, 36005, 17774, 2646, 11, 290, 318, 7151, 329, 3016, 477, 3296, 286, 3800, 290, 3159,

In [67]:
gpt2_token_ids = gpt2_encoding["input_ids"][0][:10]
gpt2_token_ids

[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690]

In [68]:
gpt2_tokenizer_ids = gpt2_encoding["input_ids"][0][:10]
gpt2_token_ids

[14247, 35030, 1690, 423, 257, 1688, 8046, 13, 484, 1690]

In [69]:
gpt2_tokenizer.decode(gpt2_token_ids)

'stage adaptations often have a major fault. they often'

Wordpiece is used by models like BERT

In [70]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_encoding = bert_tokenizer(train_reviews[:3],padding=True,
truncation=True, max_length=500, return_tensors="pt")

In [71]:
bert_encoding["input_ids"]

tensor([[  101,  2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,
          2411,  2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,
          2872,  2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,
          1000,  1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,
          1996,  2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,
          3861,  3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,
          1005,  1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,
          1012,  1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,
          3243,  8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,
          1012, 19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,
          2028,  1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,
          7987,  1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,
          1045,  1005,  2310,  2196,  2464, 11209, 2

In [72]:
bert_encoding["attention_mask"]

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 

In [73]:
bert_encoding  = bert_tokenizer(train_reviews[:3], padding=True,
                                truncation=True, max_length=500,
                                add_special_tokens=False, return_tensors="pt")

In [74]:
bert_encoding["input_ids"]

tensor([[ 2754, 17241,  2411,  2031,  1037,  2350,  6346,  1012,  2027,  2411,
          2272,  2041,  2559,  2066,  1037,  2143,  4950,  2001,  3432,  2872,
          2006,  1996,  2754,  1006,  2107,  2004,  1000,  2305,  2388,  1000,
          1007,  1012, 11430, 11320, 11368,  1005,  1055,  3257,  7906,  1996,
          2143,  4142,  1010,  2029,  2003,  2926,  3697,  2144,  1996,  3861,
          3253,  2032,  2053,  2613,  4119,  1012,  2145,  1010,  2009,  1005,
          1055,  3835,  2000,  2298,  2012,  2005,  2054,  2009,  2003,  1012,
          1996,  6370,  2090,  2745, 19881,  1998,  5696, 20726,  2003,  3243,
          8235,  1012,  1996, 10949,  1997,  2037,  3276,  2024, 11341,  1012,
         19881,  2003, 10392,  2004,  2467,  1010,  1998, 20726,  4152,  2028,
          1997,  2010,  2261,  9592,  2000,  2428,  2552,  1012,  1026,  7987,
          1013,  1028,  1026,  7987,  1013,  1028,  1045, 18766,  2008,  1045,
          1005,  2310,  2196,  2464, 11209, 20206,  

Unigram LM is used by models like ALBERT, T5, and XLM-R.

In [75]:
albert_tokenizer = transformers.AutoTokenizer.from_pretrained("albert-base-v2")
albert_encoding = albert_tokenizer(
    train_reviews[:3], padding=True, truncation=True, max_length=500,
    add_special_tokens=False, return_tensors="pt")
albert_token_ids  = albert_encoding["input_ids"]
albert_token_ids

tensor([[  876,  5004,    18,   478,    57,    21,   394,  4173,     9,    59,
           478,   340,    70,   699,   101,    21,   171,  3336,    23,  1659,
          1037,    27,    14,   876,    13,     5,  4289,    28,    13,     7,
          4893,   449,     7,     6,     9, 12508,  1612,  5909,    22,    18,
          1400,  8968,    14,   171,  2481,    15,    56,    25,  1118,  1956,
           179,    14,  2151,  1434,    61,    90,   683,  2404,     9,   174,
            15,    32,    22,    18,  2210,    20,   361,    35,    26,    98,
            32,    25,     9,    14,  5427,   128,   832, 22427,    17,  4479,
         24604,    25,  1450,  7472,     9,    14, 12289,    16,    66,  1429,
            50, 12891,     9, 22427,    25, 10356,    28,   550,    15,    17,
         24604,  3049,    53,    16,    33,   310, 11285,    20,   510,   601,
             9,     1,  5145,    13,   118,     1,  5145,    13,   118,     1,
            49, 14586,    30,    31,    22,   195,  

In [76]:
hf_tokenizer = transformers.PreTrainedTokenizerFast(
    tokenizer_object=bpe_tokenizer
)
hf_encodings = hf_tokenizer(train_reviews[:3], padding=True, truncation=True,
                            max_length=500, return_tensors="pt")
hf_encodings["input_ids"]

tensor([[159, 402, 176, 246,  61, 782, 156, 737, 252,  42, 239,  51, 154, 460,
         917,  17, 272, 156, 737, 576, 215, 976, 275,  42, 199,  44, 554,  42,
         192, 585,  57, 160, 259, 170, 157, 143, 138, 159, 402,  11, 589, 152,
           5, 819, 168, 230,   5, 521, 924, 981, 962, 250,  61,  10,  60, 426,
         526, 959,  60, 138, 199, 150, 319,  15, 363, 141, 957, 694,  47, 696,
          61, 875, 138, 960, 337, 414, 140, 157, 385, 174, 433, 161, 221, 145,
         213,  17, 549,  15, 151,  10,  60,  55, 416, 146, 407, 144, 182, 303,
         151, 141,  17, 138, 547, 538, 528, 768,  54, 335,  42, 203,  44, 270,
          46, 153, 876, 141, 919, 233, 522, 172, 141, 719, 162, 807, 279,  17,
         138,  45,  66,  55, 188, 989, 156, 378, 698, 301, 296, 689, 212, 558,
         926, 148,  17,  44, 270,  46, 141,  47, 279, 302, 171, 152, 787,  15,
         153, 522, 172, 766, 205, 156, 234, 677, 161, 139, 513, 146, 370, 251,
         219, 162, 197, 162, 166,  50, 265,  47, 266

Building and Training a Sentiment Analysis Model

In [77]:
def collate_fn(batch, tokenizer=bert_tokenizer):
    reviews = [review["text"] for review in batch]
    labels = [[review["label"]] for review in batch]
    encodings = tokenizer(reviews, padding=True, truncation=True,
                          max_length=200, return_tensors="pt")
    labels = torch.tensor(labels, dtype=torch.float32)
    return encodings, labels

batch_size = 256
imdb_train_loader = DataLoader(imdb_train_set, batch_size=batch_size,
                               collate_fn=collate_fn, shuffle=True)
imdb_valid_loader = DataLoader(imdb_valid_set, batch_size=batch_size,
                               collate_fn=collate_fn)
imdb_test_loader = DataLoader(imdb_test_set, batch_size=batch_size,
collate_fn=collate_fn)


In [78]:
class SentimentalAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64,
                 pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        _outputs, hidden_states = self.gru(embeddings)
        return self.output(hidden_states[-1])

In [79]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

sequences = torch.tensor([[1, 2, 0, 0], [5, 6, 7, 8]])
packed = pack_padded_sequence(sequences, lengths=(2, 4),
                              enforce_sorted=False, batch_first=True)
packed

PackedSequence(data=tensor([5, 1, 6, 2, 7, 8]), batch_sizes=tensor([2, 2, 1, 1]), sorted_indices=tensor([1, 0]), unsorted_indices=tensor([1, 0]))

In [80]:
padded, lengths = pad_packed_sequence(packed, batch_first=True)
padded, lengths

(tensor([[1, 2, 0, 0],
         [5, 6, 7, 8]]),
 tensor([2, 4]))

In [81]:
class SentimentalAnalysisModelPackedSeq(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128,
                 hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim,
                                  padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        return self.output(hidden_states[-1])

In [82]:
torch.manual_seed(42)

vocab_size = bert_tokenizer.vocab_size
imdb_model_ps = SentimentalAnalysisModelPackedSeq(vocab_size).to(device)
n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_ps.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)
# history = train(imdb_model_ps, optimizer, xentropy, accuracy,
#                 imdb_train_loader, imdb_valid_loader, n_epochs)

In [83]:
class SentimentAnalysisModelBidi(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128,
                 hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, 
                                  padding_idx=pad_id)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        self.output = nn.Linear(2 * hidden_dim , 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
        return self.output(top_states)

In [84]:
torch.manual_seed(42)

vocab_size  = bert_tokenizer.vocab_size
imdb_model_bidi = SentimentAnalysisModelBidi(vocab_size).to(device)

n_epochs=10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bidi.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

# history = train(imdb_model_bidi, optimizer, xentropy, accuracy, imdb_train_loader,
# imdb_valid_loader, n_epochs)

In [85]:
Out.clear()
del_vars(["albert_token_ids", "attention_mask", "bpe_batch_ids", "encoded_text",
          "lengths", "optimizer", "padded", "probs", "samples", "sequences",
          "x", "xentropy", "y", "Y_logits"])

Reusing Pretrained Embeddings and Language Models

In [86]:
bert_model = transformers.AutoModel.from_pretrained("bert-base-uncased")
bert_model.embeddings.word_embeddings

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 248.53it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding(30522, 768, padding_idx=0)

In [87]:
class SentimentAnalysisModelPreEmbeds(nn.Module):
    def __init__(self, pretrained_embeddings, n_layers=2, hidden_dim=64,
                 dropout=0.2):
        super().__init__()
        weights = pretrained_embeddings.weight.data
        self.embed = nn.Embedding.from_pretrained(weights, freeze=True)
        embed_dim = weights.shape[-1]
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
                          batch_first=True, dropout=dropout, bidirectional=True)
        self.output = nn.Linear(2 * hidden_dim, 1)

    def forward(self, encodings):
        embeddings = self.embed(encodings["input_ids"])
        lengths  = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(embeddings, lengths=lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        n_dims = self.output.in_features
        top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
        return self.output(top_states)

In [88]:
torch.manual_seed(42)

imdb_model_bert_embeds = SentimentAnalysisModelPreEmbeds(
    bert_model.embeddings.word_embeddings).to(device)

n_epochs = 10
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert_embeds.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

# history = train(imdb_model_bert_embeds, optimizer, xentropy, accuracy,
#                 imdb_train_loader, imdb_valid_loader, n_epochs)

In [89]:
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True,
max_length = 200, truncation=True,
return_tensors="pt")
bert_output = bert_model(**bert_encoding)
bert_output.last_hidden_state.shape

torch.Size([3, 200, 768])

In [90]:
del_vars(["bert_model"])

In [91]:
class SentimentAnalysisModelBert(nn.Module):
    def __init__(self, n_layers=2, hidden_dim=64, dropout=0.2):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        embed_dim = self.bert.config.hidden_size
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers,
        batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1)

    def forward(self, encodings):
        contextualized_embeddings = self.bert(**encodings).last_hidden_state
        lengths = encodings["attention_mask"].sum(dim=1)
        packed = pack_padded_sequence(contextualized_embeddings, lengths=lengths.cpu(),
        batch_first= True, enforce_sorted=False)
        _outputs, hidden_states = self.gru(packed)
        return self.output(hidden_states[-1])


In [92]:
torch.manual_seed(42)

imdb_model_bert = SentimentAnalysisModelBert().to(device)
imdb_model_bert.bert.requires_grad_(False)

n_epochs = 4
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

# history = train(imdb_model_bert, optimizer, xentropy, accuracy,
#                 imdb_train_loader, imdb_valid_loader, n_epochs)
                

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1288.91it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [93]:
del_vars(["imdb_model_bert"])

In [94]:
class SentimentAnalysinModelBert2(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output  = nn.Linear(self.bert.config.hidden_size,1)

    def forward(self, encodings):
        bert_output= self.bert(**encodings)
        return self.output(bert_output.last_hidden_state[:, 0])

In [95]:
class SentimentAnalysisModelBert3(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.bert = transformers.AutoModel.from_pretrained("bert-base-uncased")
        self.output = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, encodings):
        bert_output = self.bert(**encodings)
        return self.output(bert_output.pooler_output)

In [96]:
torch.manual_seed(42)

imdb_model_bert3 = SentimentAnalysisModelBert3().to(device)
imdb_model_bert3.bert.requires_grad_(False)

n_epochs = 5
xentropy = nn.BCEWithLogitsLoss()
optimizer = torch.optim.NAdam(imdb_model_bert3.parameters())
accuracy = torchmetrics.Accuracy(task="binary").to(device)

# history = train(imdb_model_bert3, optimizer, xentropy, accuracy,
#                 imdb_train_loader, imdb_valid_loader, n_epochs)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1144.70it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [97]:
imdb_model_bert3.bert.pooler.requires_grad_(True)

# history = train(imdb_model_bert3, optimizer, xentropy, accuracy,
#                 imdb_train_loader, imdb_valid_loader, n_epochs)

BertPooler(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (activation): Tanh()
)

In [98]:
del_vars(["imdb_model_bert3"])

In [99]:
from transformers import BertForSequenceClassification

torch.manual_seed(42)
bert_for_binary_clf =  BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2, dtype=torch.float16
).to(device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1279.24it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [100]:
encoding = bert_tokenizer(["This was a great movie!"])
with torch.no_grad():
    output = bert_for_binary_clf(
        input_ids=torch.tensor(encoding["input_ids"], device=device),
        attention_mask = torch.tensor(encoding["attention_mask"], device=device)
    )
output.logits

tensor([[0.2465, 0.3496]], device='cuda:0', dtype=torch.float16)

In [101]:
torch.softmax(output.logits, dim=-1)

tensor([[0.4741, 0.5259]], device='cuda:0', dtype=torch.float16)

In [102]:
with torch.no_grad():
    output = bert_for_binary_clf(
        input_ids=torch.tensor(encoding["input_ids"], device=device),
        attention_mask=torch.tensor(encoding["attention_mask"], device=device),
        labels=torch.tensor([1], device=device))

output.loss

tensor(0.6431, device='cuda:0', dtype=torch.float16)

The Trainer API

We could use our own training function to train this model, or we can use the Trainer API instead.

The Trainer API needs datasets that are already tokenized, so let's prepare them:

In [103]:
def tokenize_batch(batch):
    return bert_tokenizer(batch["text"], truncation=True, max_length=200)

tok_imdb_train_set = imdb_train_set.map(tokenize_batch, batched=True)
tok_imdb_valid_set = imdb_valid_set.map(tokenize_batch, batched=True)
tok_imdb_test_set  = imdb_test_set.map(tokenize_batch, batched=True)


The Trainer API does'nt support streaming metrics so we need to write our own evaluation function:

In [104]:
def compute_accuracy(pred):
    return {"accuracy": (pred.label_ids == pred.predictions.argmax(-1)).mean()}


In [105]:
from transformers import TrainingArguments

train_args = TrainingArguments(
    output_dir="my_imdb_model", num_train_epochs=2,
    per_device_train_batch_size=128, per_device_eval_batch_size=128,
    eval_strategy="epoch", logging_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="accuracy",
    report_to="none"
)

In [108]:
from transformers import DataCollatorWithPadding, Trainer

trainer = Trainer(
    bert_for_binary_clf, train_args,train_dataset=tok_imdb_train_set,
    eval_dataset=tok_imdb_valid_set, compute_metrics=compute_accuracy,
    data_collator=DataCollatorWithPadding(bert_tokenizer)
)
# train_output = trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [109]:
del_vars(["bert_for_binary_clf"])

Pipelines

In [110]:
from transformers import pipeline

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
classifier_imdb = pipeline("sentiment-analysis", model=model_name,
                           truncation=True, max_length=512)
classifier_imdb(train_reviews[:10])

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 944.62it/s]


[{'label': 'POSITIVE', 'score': 0.9996108412742615},
 {'label': 'POSITIVE', 'score': 0.9998623132705688},
 {'label': 'NEGATIVE', 'score': 0.9943684935569763},
 {'label': 'POSITIVE', 'score': 0.997913658618927},
 {'label': 'POSITIVE', 'score': 0.999544084072113},
 {'label': 'NEGATIVE', 'score': 0.9845332503318787},
 {'label': 'POSITIVE', 'score': 0.9859278202056885},
 {'label': 'POSITIVE', 'score': 0.9993758797645569},
 {'label': 'POSITIVE', 'score': 0.9978922009468079},
 {'label': 'NEGATIVE', 'score': 0.9997020363807678}]

In [112]:
accuracy = torchmetrics.Accuracy(task="binary").to(device)
with torch.no_grad():
    accuracy.reset()
    text_imdb_valid_loader  = DataLoader(imdb_valid_set, batch_size=256)
    for index, batch in enumerate(text_imdb_valid_loader):
        y_pred = classifier_imdb(batch["text"], truncation=True)
        y_pred = torch.tensor([pred["label"] == "positive" for pred in y_pred], dtype=int)
        accuracy.update(y_pred, batch["label"])
        print(f"\r{index+1}/{len(text_imdb_valid_loader)}", end="")

accuracy.compute()

20/20

tensor(0.4988, device='cuda:0')

Models can be very biased. For example, it may like or dislike some countries depending on the data it was trained on, and how it is used, so use it with care:

In [113]:
countries = ["Iraq", "India", "the USA", "Vietnam"]
texts = [f"I am from{country}" for country in countries]
list(zip(countries, classifier_imdb(texts)))

[('Iraq', {'label': 'POSITIVE', 'score': 0.8410910367965698}),
 ('India', {'label': 'NEGATIVE', 'score': 0.9836459159851074}),
 ('the USA', {'label': 'POSITIVE', 'score': 0.9605668187141418}),
 ('Vietnam', {'label': 'POSITIVE', 'score': 0.7017163634300232})]

In [114]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"
classifier_imdb_with_neutral = pipeline("sentiment-analysis", model=model_name)
list(zip(countries, classifier_imdb_with_neutral(texts)))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6254.72it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[('Iraq', {'label': 'neutral', 'score': 0.8552864789962769}),
 ('India', {'label': 'neutral', 'score': 0.7932306528091431}),
 ('the USA', {'label': 'neutral', 'score': 0.8271870613098145}),
 ('Vietnam', {'label': 'neutral', 'score': 0.8758615851402283})]

In [115]:
model_name = "huggingface/distilbert-base-uncased-finetuned-mnli"
classifier_mnli = pipeline("text-classification", model=model_name)
classifier_mnli([
    "She loves me. [SEP] She loves me not. [SEP]",
    "Alice just woke up. [SEP] Alice is awake. [SEP]",
    "I like dogs. [SEP] Everyone likes dogs. [SEP]"])

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1098.44it/s]


[{'label': 'contradiction', 'score': 0.9717152714729309},
 {'label': 'entailment', 'score': 0.9119168519973755},
 {'label': 'neutral', 'score': 0.9509281516075134}]

In [116]:
Out.clear()
del_vars(["classifier_imdb", "clssifier_mnli", "classifier_imdb_with_neutral",
          "trainer"])

An Encoder- Decoder Network for Neural Machine
Translation